# v3.0.0 PD AST Spectrograms — All Tasks Sensitivity Analysis

All-tasks sensitivity analysis for PD screening on Bridge2AI v3.0.0.
Instead of the 8 selected tasks used in the primary experiment, this notebook
uses **ALL available tasks** in the spectrogram data.

Adapted from v2.0.1 all-tasks pipeline with v3 data changes:
- Path: `features/torchaudio_mel_spectrogram.parquet` (column: `mel_spectrogram`)
- Mel bins: 60 (resized to 128 for AST)
- Hierarchical phenotype: `phenotype/diagnosis/parkinsons_disease.tsv` + `control.tsv`
- Participant IDs: 6-digit numeric (zero-padded)

Training changes for larger dataset (vs selected-tasks):
- Maximum epochs: 20 (reduced from 30)
- Early stopping patience: 5 epochs (reduced from 10)
- No learning rate scheduler (constant LR throughout)
- Model selection threshold: 1e-6 (relaxed from 0.01)
- Class weights: [1.0, N_neg/N_pos] (unit weight for majority, proportional for minority)

In [ ]:
#1 - Imports & data loading
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from tqdm import tqdm
from scipy.ndimage import zoom
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import ASTModel, ASTConfig

ROOT = Path('/data0/b2ai-voice/3.0.0')
SPEC_PATH = ROOT / 'features' / 'torchaudio_mel_spectrogram.parquet'
PD_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'parkinsons_disease.tsv'
CTRL_PHEN = ROOT / 'phenotype' / 'diagnosis' / 'control.tsv'
RESULTS_DIR = Path('/home/saptpurk/bridge2ai-voice-parkinsons-ast/results/v3')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MIN_TIME_FRAMES = 100

# Load spectrograms (ALL tasks, no task filter)
pf = pq.ParquetFile(SPEC_PATH)
parts = []
for i in range(pf.num_row_groups):
    parts.append(pf.read_row_group(i, columns=['participant_id','session_id','task_name','mel_spectrogram','n_frames']).to_pandas())
spec_df = pd.concat(parts, ignore_index=True)
spec_df['participant_id'] = spec_df['participant_id'].astype(str).str.zfill(6)

# Build PD labels
pd_pheno = pd.read_csv(PD_PHEN, sep='\t')
ctrl_pheno = pd.read_csv(CTRL_PHEN, sep='\t')
pd_ids = set(pd_pheno['participant_id'].astype(str).str.zfill(6))
ctrl_ids = set(ctrl_pheno['participant_id'].astype(str).str.zfill(6)) - pd_ids

spec_df['label'] = np.nan
spec_df.loc[spec_df['participant_id'].isin(pd_ids), 'label'] = 1
spec_df.loc[spec_df['participant_id'].isin(ctrl_ids), 'label'] = 0
data = spec_df.dropna(subset=['label']).copy()
data['label'] = data['label'].astype(int)
data = data[data['n_frames'] >= MIN_TIME_FRAMES].copy()

print(f'Total tasks: {data["task_name"].nunique()}')
print(f'PD: {(data["label"]==1).sum()} recs from {data[data["label"]==1]["participant_id"].nunique()} participants')
print(f'Ctrl: {(data["label"]==0).sum()} recs from {data[data["label"]==0]["participant_id"].nunique()} participants')
print(f'Total participants: {data["participant_id"].nunique()}')

In [ ]:
#2 - Spectrogram processing: (60, T) -> (128, 1024)
TARGET_FREQ = 128
TARGET_TIME = 1024

def process_spectrogram(spec_raw, target_freq=128, target_time=1024):
    spec = np.stack(spec_raw).astype(np.float32)
    n_mel, time_len = spec.shape
    if time_len < target_time:
        spec = np.pad(spec, ((0, 0), (0, target_time - time_len)), mode='reflect')
    elif time_len > target_time:
        start = (time_len - target_time) // 2
        spec = spec[:, start:start + target_time]
    freq_ratio = target_freq / n_mel
    spec = zoom(spec, (freq_ratio, 1.0), order=1)
    return spec

X_list = []
for idx, row in tqdm(data.iterrows(), total=len(data), desc='Processing spectrograms'):
    X_list.append(process_spectrogram(row['mel_spectrogram'], TARGET_FREQ, TARGET_TIME))

X_raw = np.stack(X_list)
y_raw = data['label'].values
participants_raw = data['participant_id'].values
print(f'Shape: {X_raw.shape}')

In [ ]:
#3 - Model definition (same ASTClassifier as selected-tasks)
class ASTClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained_tag='MIT/ast-finetuned-audioset-10-10-0.4593'):
        super().__init__()
        self.ast = ASTModel.from_pretrained(pretrained_tag)
        hidden = self.ast.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, pixel_values):
        out = self.ast(pixel_values).last_hidden_state[:, 0, :]
        return self.classifier(out)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

class SpectrogramDataset(torch.utils.data.Dataset):
    def __init__(self, X, y, participants, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.participants = np.array(participants)
        self.augment = augment
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.random() < 0.5:
                t = np.random.randint(50, 150)
                t0 = np.random.randint(0, max(1, x.shape[1] - t))
                x[:, t0:t0+t] = 0
            if np.random.random() < 0.5:
                f = np.random.randint(10, 30)
                f0 = np.random.randint(0, max(1, x.shape[0] - f))
                x[f0:f0+f, :] = 0
        x_3ch = x.unsqueeze(0).expand(3, -1, -1)
        return {'pixel_values': x_3ch.permute(0, 2, 1), 'labels': self.y[idx], 'participant': self.participants[idx]}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
#4 - 5-Fold Cross-Validation (adjusted hyperparams for all-tasks)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve
import copy, time

unique_participants = np.unique(participants_raw)
participant_labels = np.array([y_raw[participants_raw == p][0] for p in unique_participants])
print(f'Participants: {len(unique_participants)}, PD: {int(participant_labels.sum())}, Ctrl: {int((1-participant_labels).sum())}')

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
all_oof_probs = np.zeros(len(unique_participants), dtype=np.float32)
all_oof_labels = participant_labels.astype(np.int64).copy()

def evaluate_fold(model, loader, device):
    model.eval()
    all_probs, all_labels, all_parts = [], [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(batch['pixel_values'].to(device))
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs); all_labels.extend(batch['labels'].numpy()); all_parts.extend(batch['participant'])
    all_probs, all_labels, all_parts = np.array(all_probs), np.array(all_labels), np.array(all_parts)
    unique_parts = np.unique(all_parts)
    part_probs = np.array([all_probs[all_parts == p].mean() for p in unique_parts])
    part_labels = np.array([all_labels[all_parts == p][0] for p in unique_parts])
    return part_probs, part_labels, unique_parts

total_start = time.time()
for fold, (train_idx, val_idx) in enumerate(skf.split(unique_participants, participant_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    train_parts = unique_participants[train_idx]; val_parts = unique_participants[val_idx]
    train_mask = np.isin(participants_raw, train_parts); val_mask = np.isin(participants_raw, val_parts)

    X_tr = X_raw[train_mask]; y_tr = y_raw[train_mask]; p_tr = participants_raw[train_mask]
    X_va = X_raw[val_mask]; y_va = y_raw[val_mask]; p_va = participants_raw[val_mask]

    # Fold-specific normalization
    fold_mean, fold_std = X_tr.mean(), X_tr.std()
    X_tr_n = (X_tr - fold_mean) / (fold_std + 1e-8)
    X_va_n = (X_va - fold_mean) / (fold_std + 1e-8)

    train_ds = SpectrogramDataset(X_tr_n, y_tr, p_tr, augment=True)
    val_ds = SpectrogramDataset(X_va_n, y_va, p_va, augment=False)

    # All-tasks class weights: [1.0, N_neg/N_pos]
    cc = np.bincount(y_tr)
    cw_sampler = (1.0 / cc)[y_tr]
    cw_loss = np.array([1.0, cc[0] / cc[1]], dtype=np.float32)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=8,
        sampler=torch.utils.data.WeightedRandomSampler(cw_sampler, len(cw_sampler)),
        num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    model = ASTClassifier(num_classes=2).to(device)
    backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n]
    head_params = [p for n, p in model.named_parameters() if 'classifier' in n]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 5e-6},
        {'params': head_params, 'lr': 5e-4}
    ], weight_decay=0.01)
    # No scheduler for all-tasks (constant LR)
    criterion = FocalLoss(gamma=2.0, weight=torch.tensor(cw_loss, dtype=torch.float32).to(device))

    best_score, best_state, patience_counter = 0, None, 0
    MAX_EPOCHS, PATIENCE = 20, 5  # Reduced for larger dataset
    for epoch in range(MAX_EPOCHS):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch['pixel_values'].to(device)), batch['labels'].to(device))
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        pp, pl, _ = evaluate_fold(model, val_loader, device)
        if len(np.unique(pl)) > 1:
            auc = roc_auc_score(pl, pp)
            fpr, tpr, thr = roc_curve(pl, pp)
            f1_opt = f1_score(pl, (pp >= thr[np.argmax(tpr-fpr)]).astype(int), zero_division=0)
        else: auc, f1_opt = 0.5, 0.0
        score = 0.4*auc + 0.6*f1_opt
        if score > best_score + 1e-6:  # Relaxed threshold
            best_score = score; best_state = copy.deepcopy(model.state_dict()); patience_counter = 0
        else: patience_counter += 1
        if patience_counter >= PATIENCE: break

    model.load_state_dict(best_state)
    pp, pl, vpids = evaluate_fold(model, val_loader, device)
    for i, pid in enumerate(vpids):
        all_oof_probs[np.where(unique_participants == pid)[0][0]] = pp[i]
    fold_auc = roc_auc_score(pl, pp)
    fpr, tpr, thr = roc_curve(pl, pp); ot = thr[np.argmax(tpr-fpr)]
    po = (pp >= ot).astype(int)
    fold_results.append({'fold': fold+1, 'auc': float(fold_auc),
        'f1_opt': float(f1_score(pl, po, zero_division=0)),
        'recall_opt': float(recall_score(pl, po, zero_division=0)),
        'precision_opt': float(precision_score(pl, po, zero_division=0))})
    print(f'Fold {fold+1}: AUC={fold_auc:.4f}, F1={fold_results[-1]["f1_opt"]:.4f}')
    del model; torch.cuda.empty_cache()

print(f'\nTotal time: {(time.time()-total_start)/60:.1f} min')

In [ ]:
#5 - Summary
from scipy import stats

aucs = [r['auc'] for r in fold_results]
oof_auc = roc_auc_score(all_oof_labels, all_oof_probs)
fpr, tpr, thr = roc_curve(all_oof_labels, all_oof_probs)
oof_thresh = thr[np.argmax(tpr-fpr)]
oof_preds = (all_oof_probs >= oof_thresh).astype(int)

mean_auc = np.mean(aucs)
std_auc = np.std(aucs, ddof=1)
ci = stats.t.interval(0.95, df=len(aucs)-1, loc=mean_auc, scale=std_auc/np.sqrt(len(aucs)))

print(f'=== All-Tasks PD Sensitivity Analysis (v3.0.0) ===')
print(f'Participants: {len(unique_participants)} (PD: {int(participant_labels.sum())}, Ctrl: {int((1-participant_labels).sum())})')
print(f'Recordings: {len(X_raw)}')
print(f'Case prevalence: {participant_labels.mean()*100:.1f}%')
print(f'Mean fold AUC: {mean_auc:.4f} +/- {std_auc:.4f}')
print(f'95% CI: [{max(0, ci[0]):.4f}, {min(1, ci[1]):.4f}]')
print(f'OOF AUC: {oof_auc:.4f}')
print(f'OOF F1:  {f1_score(all_oof_labels, oof_preds, zero_division=0):.4f}')
print(f'OOF Recall: {recall_score(all_oof_labels, oof_preds, zero_division=0):.4f}')
print(f'OOF Precision: {precision_score(all_oof_labels, oof_preds, zero_division=0):.4f}')

np.savez(str(RESULTS_DIR / 'ast_pd_v3_alltasks_cv_results.npz'),
    oof_probs=all_oof_probs, oof_labels=all_oof_labels, participant_ids=unique_participants,
    fold_aucs=np.array(aucs), oof_auc=np.array(oof_auc))
print(f'Saved: {RESULTS_DIR}/ast_pd_v3_alltasks_cv_results.npz')